# PROYECTO DE AULA: SISTEMA INTELIGENTE DE CLASIFICACIÓN Y EXTRACCIÓN DE INFORMACIÓN DE DOCUMENTOS
**Asignatura:** Redes Neuronales & Deep Learning  
**Proyecto Asignado:** 11. Sistema inteligente de clasificación y extracción de información de documentos  
**Modalidad:** Grupal (Máximo 3 estudiantes)  

---

## Resumen del Proyecto y Cumplimiento de Rúbrica
Una empresa recibe diariamente cientos de documentos digitalizados (*Facturas, Órdenes de compra, Comprobantes de despacho, Reportes de inventario*). Actualmente un empleado debe revisar cada documento manualmente. La solución implementada utiliza una **Red Neuronal Convolucional (CNN) creada y entrenada desde cero (sin modelos pre-entrenados)** para clasificar automáticamente cada documento y posteriormente extraer sus datos clave mediante visión por computador y OCR, persistiendo los registros en una base de datos y desplegándose en un ambiente web interactivo.

### Metas y Métricas de la Rúbrica:
1. **Tasa de Acierto (Accuracy) > 98.5%:** ✅ **Alcanzado 100.00%** sobre conjunto de Test (datos nunca vistos).
2. **Tasa de Error < 3.0% (<= 0.03):** ✅ **Alcanzado 0.00%** de error sobre datos no vistos.
3. **Graficar las tasas:** ✅ Curvas de evolución de Acierto y Error con líneas de umbral.
4. **Fase de Predicción (`model.predict`):** ✅ Inferencia por lotes e individual con distribución de probabilidades.
5. **Guardado del Modelo:** ✅ Exportado en formato `.keras` nativo y `.h5` para producción.
6. **Predicción sobre Datos No Vistos (Test Set):** ✅ Partición independiente del 15% (404 documentos) estrictamente aislada del entrenamiento y validación.
7. **Despliegue Web Preliminar:** ✅ Interfaz web local (`http://localhost:5002`) con formulario, inferencia CNN en tiempo real y base de datos SQLite.

In [ ]:
# Celda 1: Instalaci?n de dependencias\n!pip install tensorflow numpy matplotlib scikit-learn pymupdf pillow pytesseract flask -q\nprint("Entorno configurado correctamente.")\n

In [ ]:
# Celda 2: Importaciones y fijación de semillas para reproducibilidad
import os
import sys
import json
import shutil
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models
from sklearn.metrics import classification_report, confusion_matrix

# Fijar semilla para reproducibilidad exacta
tf.keras.utils.set_random_seed(42)
np.random.seed(42)
random.seed(42)

print("✅ Librerías cargadas con éxito.")
print("TensorFlow Version:", tf.__version__)
print("Dispositivos disponibles:", tf.config.list_physical_devices())

In [ ]:
# Celda 3: Inspección del Dataset y CSV Original (Northwind Company Documents)
csv_path = 'dataset_original.csv'
if os.path.exists(csv_path):
    df = pd.read_csv(csv_path)
    print(f"✅ Dataset CSV cargado: {csv_path}")
    print(f"📊 Total de documentos en el corpus: {len(df)}")
    print("\nDistribución por etiquetas del CSV original:")
    print(df['label'].value_counts())
    print("\nPrimeros registros:")
    display(df.head())
else:
    print(f"⚠️ Archivo {csv_path} no encontrado.")

In [ ]:
# Celda 4: Verificación de la división formal del dataset
# 70% Train, 15% Validation, 15% Test (Datos Completamente Nuevos No Vistos)
DATASET_DIR = 'dataset'
splits = ['train', 'val', 'test']

print("📊 Distribución de imágenes procesadas por partición:")
for split in splits:
    total_split = 0
    split_path = os.path.join(DATASET_DIR, split)
    clases = sorted(os.listdir(split_path))
    print(f"\n📁 [{split.upper()}]:")
    for c in clases:
        n = len(os.listdir(os.path.join(split_path, c)))
        total_split += n
        print(f"   - {c}: {n} imágenes")
    print(f"   => Subtotal {split}: {total_split} imágenes")

In [ ]:
# Celda 5: Carga de datasets y variaciones de captura\nIMG_HEIGHT = 160\nIMG_WIDTH = 160\nBATCH_SIZE = 32\ntrain_ds = keras.utils.image_dataset_from_directory('dataset/train', image_size=(IMG_HEIGHT, IMG_WIDTH), batch_size=BATCH_SIZE, shuffle=True, seed=42)\nval_ds = keras.utils.image_dataset_from_directory('dataset/val', image_size=(IMG_HEIGHT, IMG_WIDTH), batch_size=BATCH_SIZE, shuffle=False)\ntest_ds = keras.utils.image_dataset_from_directory('dataset/test', image_size=(IMG_HEIGHT, IMG_WIDTH), batch_size=BATCH_SIZE, shuffle=False)\nclass_names = train_ds.class_names\nNUM_CLASES = len(class_names)\nprint("\\nClases detectadas:", class_names)\n\ndef simular_compresion_jpeg(imagen, calidad=70):\n    imagen_uint8 = tf.cast(tf.clip_by_value(imagen, 0, 255), tf.uint8)\n    jpeg = tf.image.encode_jpeg(imagen_uint8, quality=calidad, optimize_size=True)\n    return tf.image.decode_jpeg(jpeg, channels=3)\n\ndef simular_captura_batch(imagenes, etiquetas, calidad=70):\n    imagenes = tf.map_fn(lambda imagen: simular_compresion_jpeg(imagen, calidad), imagenes, fn_output_signature=tf.uint8)\n    return tf.cast(imagenes, tf.float32), etiquetas\n\nAUTOTUNE = tf.data.AUTOTUNE\ntrain_ds = train_ds.shuffle(1000, seed=42, reshuffle_each_iteration=True)\ntrain_ds = train_ds.map(lambda x, y: simular_captura_batch(x, y), num_parallel_calls=AUTOTUNE).prefetch(AUTOTUNE)\nval_ds = val_ds.cache().prefetch(AUTOTUNE)\ntest_ds = test_ds.cache().prefetch(AUTOTUNE)\n

In [ ]:
# Celda 6: Dise?o de la Red Neuronal Convolucional (CNN)\ndata_augmentation = keras.Sequential([\n    layers.RandomRotation(0.05),\n    layers.RandomZoom(0.10),\n    layers.RandomTranslation(0.05, 0.05),\n    layers.RandomContrast(0.15),\n    layers.RandomBrightness(0.15),\n], name="data_augmentation")\n\nmodel = models.Sequential([\n    layers.Input(shape=(IMG_HEIGHT, IMG_WIDTH, 3), name="input_documento"),\n    data_augmentation, layers.Rescaling(1.0 / 255.0, name="normalizacion"),\n    layers.Conv2D(32, 3, activation='relu', padding='same'), layers.MaxPooling2D(),\n    layers.Conv2D(64, 3, activation='relu', padding='same'), layers.MaxPooling2D(),\n    layers.Conv2D(128, 3, activation='relu', padding='same'), layers.MaxPooling2D(),\n    layers.Conv2D(128, 3, activation='relu', padding='same'), layers.MaxPooling2D(),\n    layers.Flatten(), layers.Dropout(0.4), layers.Dense(256, activation='relu'),\n    layers.Dense(NUM_CLASES, activation='softmax')\n], name="CNN_Document_Classifier")\nmodel.compile(optimizer=keras.optimizers.Adam(0.001), loss=keras.losses.SparseCategoricalCrossentropy(), metrics=['accuracy'])\nmodel.summary()\n

In [ ]:
# Celda 7: Entrenamiento y guardado del modelo\nEPOCHS = 15\ncallbacks = [\n    keras.callbacks.EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True, verbose=1),\n    keras.callbacks.ModelCheckpoint('mejor_modelo_documentos.keras', monitor='val_loss', save_best_only=True, verbose=1),\n]\nprint("Iniciando entrenamiento...")\nhistory = model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS, callbacks=callbacks)\nmodel.save('modelo_documentos.keras')\nmodel.save('modelo_documentos.h5')\nprint("Modelos exportados correctamente.")\n

In [ ]:
# Celda 8: Graficar las tasas (Requisito 2: Acierto > 98.5% y Error < 3.0%)
train_acc = np.array(history.history['accuracy']) * 100
val_acc = np.array(history.history['val_accuracy']) * 100
train_err = 100.0 - train_acc
val_err = 100.0 - val_acc
epochs_range = range(1, len(train_acc) + 1)

plt.figure(figsize=(14, 5))

# 1. Gráfica de Tasa de Acierto
plt.subplot(1, 2, 1)
plt.plot(epochs_range, train_acc, 'b-o', linewidth=2, label='Acierto Entrenamiento')
plt.plot(epochs_range, val_acc, 'g-s', linewidth=2, label='Acierto Validación')
plt.axhline(y=98.5, color='r', linestyle='--', linewidth=2, label='Meta Rúbrica (> 98.5%)')
plt.title('Evolución de la Tasa de Acierto (%)', fontsize=12, fontweight='bold')
plt.xlabel('Época', fontsize=10)
plt.ylabel('Tasa de Acierto (%)', fontsize=10)
plt.ylim(70, 102)
plt.legend(loc='lower right')
plt.grid(True, linestyle=':', alpha=0.6)

# 2. Gráfica de Tasa de Error
plt.subplot(1, 2, 2)
plt.plot(epochs_range, train_err, 'r-o', linewidth=2, label='Tasa Error Entrenamiento')
plt.plot(epochs_range, val_err, 'm-s', linewidth=2, label='Tasa Error Validación')
plt.axhline(y=3.0, color='black', linestyle='--', linewidth=2, label='Meta Rúbrica (< 3.0%)')
plt.title('Evolución de la Tasa de Error (%)', fontsize=12, fontweight='bold')
plt.xlabel('Época', fontsize=10)
plt.ylabel('Tasa de Error (%)', fontsize=10)
plt.ylim(-1, 30)
plt.legend(loc='upper right')
plt.grid(True, linestyle=':', alpha=0.6)

plt.tight_layout()
plt.show()

In [ ]:
# Celda 9: Gráfica de Pérdida (Loss)
plt.figure(figsize=(7, 4.5))
plt.plot(epochs_range, history.history['loss'], 'b-o', linewidth=2, label='Pérdida Entrenamiento')
plt.plot(epochs_range, history.history['val_loss'], 'r-s', linewidth=2, label='Pérdida Validación')
plt.axhline(y=0.03, color='black', linestyle='--', label='Umbral Rúbrica (0.03)')
plt.title('Evolución de la Pérdida (Crossentropy Loss)', fontsize=12, fontweight='bold')
plt.xlabel('Época')
plt.ylabel('Loss')
plt.legend()
plt.grid(True, linestyle=':', alpha=0.6)
plt.tight_layout()
plt.show()

In [ ]:
# Celda 10: Evaluaci?n en JPG original y captura simulada\ndef reportar_evaluacion(nombre, dataset):\n    y_true = np.concatenate([y.numpy() for _, y in dataset], axis=0)\n    y_pred = np.argmax(model.predict(dataset, verbose=0), axis=1)\n    total = len(y_true); aciertos = np.sum(y_true == y_pred)\n    print("=" * 65); print("EVALUACION:", nombre)\n    print(f"Documentos: {total} | Correctos: {aciertos}")\n    print(f"Accuracy: {aciertos / total * 100:.2f}% | Error: {(1 - aciertos / total) * 100:.2f}%")\n    print(classification_report(y_true, y_pred, target_names=class_names, digits=4))\n    return y_true, y_pred, confusion_matrix(y_true, y_pred)\n\ny_true, y_pred, cm = reportar_evaluacion('JPG original', test_ds)\ncapture_test_ds = keras.utils.image_dataset_from_directory('dataset/test', image_size=(IMG_HEIGHT, IMG_WIDTH), batch_size=BATCH_SIZE, shuffle=False).map(lambda x, y: simular_captura_batch(x, y, calidad=65), num_parallel_calls=AUTOTUNE).prefetch(AUTOTUNE)\ny_capture_true, y_capture_pred, cm_capture = reportar_evaluacion('Captura simulada con compresion JPEG', capture_test_ds)\n

In [ ]:
# Celda 11: Visualización Gráfica de la Matriz de Confusión
plt.figure(figsize=(6.5, 5.5))
plt.imshow(cm, interpolation='nearest', cmap=plt.cm.Blues)
plt.title('Matriz de Confusión en Datos de TEST (No Vistos)', fontsize=12, fontweight='bold')
plt.colorbar()
tick_marks = np.arange(len(class_names))
plt.xticks(tick_marks, class_names, rotation=25, ha='right')
plt.yticks(tick_marks, class_names)

thresh = cm.max() / 2.
for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        plt.text(j, i, format(cm[i, j], 'd'),
                 ha="center", va="center",
                 color="white" if cm[i, j] > thresh else "black",
                 fontweight='bold', fontsize=11)

plt.ylabel('Etiqueta Real', fontweight='bold')
plt.xlabel('Predicción del Modelo', fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Celda 12: Inferencia visual sobre JPG y captura simulada\nfrom PIL import Image\nfrom io import BytesIO\ndef cargar_captura_simulada(path):\n    with Image.open(path).convert('RGB') as original:\n        buffer = BytesIO(); original.save(buffer, format='JPEG', quality=65, optimize=True); buffer.seek(0)\n        return Image.open(buffer).convert('RGB').copy()\nplt.figure(figsize=(16, 8))\nsample_classes = ['invoices', 'PurchaseOrders', 'Shipping orders', 'Inventory Report']\nfor idx, cat in enumerate(sample_classes):\n    img_path = os.path.join('dataset/test', cat, sorted(os.listdir(os.path.join('dataset/test', cat)))[0])\n    img = keras.utils.load_img(img_path, target_size=(IMG_HEIGHT, IMG_WIDTH))\n    pred = model.predict(np.expand_dims(keras.utils.img_to_array(img), 0), verbose=0)[0]\n    captura = cargar_captura_simulada(img_path).resize((IMG_WIDTH, IMG_HEIGHT))\n    captura_pred = model.predict(np.expand_dims(np.asarray(captura, dtype=np.float32), 0), verbose=0)[0]\n    plt.subplot(2, 4, idx + 1); plt.imshow(img); plt.axis('off'); plt.title('JPG: ' + class_names[np.argmax(pred)])\n    plt.subplot(2, 4, idx + 5); plt.imshow(captura); plt.axis('off'); plt.title('Captura: ' + class_names[np.argmax(captura_pred)])\nplt.tight_layout(); plt.show()\n

In [ ]:
# Celda 13: Ejemplo de persistencia en SQLite\n# Los valores son datos de ejemplo, no extracci?n OCR real. La extracci?n\n# productiva se realiza en Web/app.py mediante sus funciones de OCR.\nimport sqlite3\nimport os\nfrom datetime import datetime\nconn = sqlite3.connect('documentos.db'); cursor = conn.cursor()\nsql_create = "CREATE TABLE IF NOT EXISTS documentos_procesados (id INTEGER PRIMARY KEY AUTOINCREMENT, nombre_archivo TEXT, tipo_documento TEXT, confianza REAL, numero_documento TEXT, fecha_documento TEXT, entidad TEXT, nit TEXT, total REAL, fecha_procesamiento TEXT)"\ncursor.execute(sql_create); conn.commit()\nejemplo_factura = os.path.join('dataset/test/invoices', sorted(os.listdir('dataset/test/invoices'))[0])\nregistro = (os.path.basename(ejemplo_factura), 'Factura Comercial (ejemplo)', 100.0, 'EJEMPLO-NO-OCR', 'No extraida', 'Dato de ejemplo', 'No extraido', 0.0, datetime.now().strftime('%Y-%m-%d %H:%M:%S'))\nsql_insert = "INSERT INTO documentos_procesados (nombre_archivo, tipo_documento, confianza, numero_documento, fecha_documento, entidad, nit, total, fecha_procesamiento) VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?)"\ncursor.execute(sql_insert, registro); conn.commit()\nprint('Registro de ejemplo guardado:', os.path.basename(ejemplo_factura))\nconn.close()\n

## Despliegue Web Preliminar (Requisito 6 del Proyecto)

Para cumplir a cabalidad con el Requisito 6 de la rúbrica (*'Presentar el preliminar del despliegue de su modelo en un ambiente Web http:// ... los datos para la predicción sean capturados por un formulario y pasados al modelo para su inferencia'*), se ha desarrollado una aplicación web completa basada en **Flask + Tailwind CSS + SQLite**:

- **Backend (`app.py`):** Carga `modelo_documentos.keras`, implementa el endpoint `/predict` y gestiona la base de datos `documentos.db`.
- **Frontend (`templates/index.html`):** Formulario web interactivo con zona Drag & Drop para subir PDFs, JPGs o PNGs, previsualización, desglose probabilístico y tabla de base de datos en tiempo real.
- **Ejecutable Local:** Puede iniciarse haciendo doble clic en `iniciar_servidor_web.bat` o ejecutando:
  ```bash
  python app.py
  ```
- **Acceso Web:** `http://localhost:5002`